In [1]:
import math_toolkit
math_toolkit.activate()

# Plotting symbolic mathematics

`math_toolkit` provides tools for visualization of mathematical expressions. Plots live in a `figure`. A figure collects the visual representations and the live notebook state around them: parameter controls, view state, status messages, and display layout.

Plotting commands accept symbolic expressions. Numerically implemented functions must first be represented as symbolic `ImplementedFunction` objects.

## Plotting commands

The current plotting commands cover these common mathematical shapes:

- `plot`: scalar functions of one variable.
- `list_plot`: finite discrete point sets and integer-indexed expressions.
- `temperature_plot`: scalar fields of two variables, shown as field values.
- `contour_plot`: scalar fields of two variables, shown as level sets.
- `domain_plot`: regions defined by signs or Boolean conditions.
- `parametric_plot`: curves defined by coordinate pairs depending on one parameter.

## First plot

A figure is the notebook container for plots. The basic pattern is to create a figure, display it, and add a symbolic expression as a curve.

`fig.show()` creates a visible display generation for the figure. Plot commands update the figure's mathematical state; the visible display follows those updates.

Calling `fig.show()` again creates another visible display of the same figure. The later invalidation section explains what happens to older displayed outputs.

In [2]:
fig = figure("plotting-guide-first-plot")
fig.show()

A figure context sets the active plotting target. Inside `with fig:`, top-level plotting commands add plots to `fig`.

This separates figure setup from the mathematical commands that add curves, fields, regions, and boundaries.

In [3]:
with fig:
    plot(sin(x), x)

## Adding another curve

A second plotting command in the same figure adds another curve. Both commands use `x` as an unbounded plotting variable, so the current figure view supplies the visible sampling window.

Several curves belong in one figure when they answer the same comparison. In this case the two elementary waves share the same independent variable and the same visible range.

In [4]:
with fig:
    plot(cos(x), x)

## Names make plots addressable

A figure name identifies the plotting surface. A plot name identifies one plotted object inside that surface.

Use `name=` when a later cell should retrieve, restyle, relabel, update, or remove the same plot. Use `label=` for the visible legend or control label.

In [5]:
named_fig = figure("plotting-guide-named-plots")
named_fig.show()

In [6]:
with named_fig:
    plot(sin(x), x, name="wave", label=r"$\sin(x)$")

In [7]:
with named_fig:
    get_plot("wave").label = "sine wave"

## Domain limits

A plotting command needs to know which symbols are plotting variables. Bounds are optional for curves, scalar fields, contours, and domains: `plot(sin(x), x)` uses the current figure view.

Finite domain tuples such as `(x, -pi, pi)` state the mathematical interval that should be sampled. They are appropriate when the object itself is being studied on a specific interval.

In [8]:
domain_fig = figure("plotting-guide-domains")
domain_fig.show()

In [9]:
with domain_fig:
    plot(sin(x) / x, (x, -12, 12), name="sinc", label=r"$\sin(x)/x$")

## Parameters

Symbols that are not plotting variables can become parameters. A parameter value supplies the numerical value used for sampling.

Automatic parameter detection is the basic form: any free symbol that is not the plotting variable is treated as a parameter when `fig.parameters(...)` supplies its value.

In [10]:
parameter_fig = figure("plotting-guide-parameters")
parameter_fig.show()

In [11]:
with parameter_fig:
    plot(exp(-x**2 / 8) * cos(a * x), x, name="damped-wave", label=r"$\text{damped wave}$")

Explicit parameter specs add control metadata. The value still supplies the numerical parameter, while `min`, `max`, and `step` describe the slider.

In [12]:
with parameter_fig:
    parameter_fig.parameters({a: {"value": 2.0, "min": 0.5, "max": 6.0, "step": 0.25}})
    plot(
        exp(-x**2 / 8) * cos(a * x),
        x,
        name="damped-wave",
        label="damped wave",
    )

## More plot types

Discrete point sets, scalar fields, level sets, regions, and parametric curves use the same figure/context pattern.

`list_plot` renders finite point sets and integer-indexed expressions. `temperature_plot` samples scalar-field values over a two-dimensional view. `contour_plot` samples the same kind of field but displays level sets. `domain_plot` fills points that satisfy signs or Boolean conditions. `parametric_plot` samples coordinate pairs over an explicit parameter interval.

In [13]:
field_fig = figure("plotting-guide-field-types")
field_fig.show()

In [14]:
field = exp(-(x**2 + y**2) / 4) * cos(2 * x) * sin(2 * y)
with field_fig:
    temperature_plot(field, (x, -3, 3), (y, -3, 3), name="field", label="field value", samples=50)
    contour_plot(field, (x, -3, 3), (y, -3, 3), name="levels", label="level curves", samples=50)

Discrete data belongs in `list_plot`. Value lists use their own zero-based integer positions, while expression list plots sample integer index values from an explicit range or from the visible x-axis view.

In [15]:
discrete_fig = figure("plotting-guide-discrete-list-plots")
discrete_fig.show()

with discrete_fig:
    list_plot([2, 3, 5, 7, 11], name="values", label="indexed values")
    list_plot(n**2, (n, 0, 8), name="squares", label="integer squares")

In [16]:
region_fig = figure("plotting-guide-regions")
region_fig.show()

In [17]:
with region_fig:
    domain_plot(1 - x**2 - y**2, (x, -1.5, 1.5), (y, -1.5, 1.5), name="disk", label="unit disk", samples=60)
    parametric_plot((cos(t), sin(t)), (t, 0, 2 * pi), name="boundary", label="boundary", samples=200)

## Numeric functions

Plotting commands work in the symbolic layer. A numerical Python implementation must therefore be wrapped as an `ImplementedFunction` before it can appear in a plotted symbolic expression.

The wrapper gives the numerical implementation a symbolic function head. The plotted expression remains symbolic, while numerical sampling calls the implementation behind that head.

In [18]:
numeric_fig = figure("plotting-guide-numeric-functions")
numeric_fig.show()

In [19]:
import numpy as np
def gaussian_numeric(z):
    return np.exp(-z**2)
G = ImplementedFunction("G", gaussian_numeric)

with numeric_fig:
    plot(G(x), (x, -3, 3), name="gaussian", label="G(x)")

## Superposition

One figure can contain several compatible plot types. A heatmap and its contours describe the same scalar field in different ways. A region and a parametric curve can place a filled set and its boundary in the same coordinates.

Superposition is most readable when the figure has one mathematical purpose and each layer contributes a distinct part of that purpose.

In [20]:
mixed_fig = figure("plotting-guide-superposition")
mixed_fig.show()

In [21]:
potential = x**2 + y**2 - 1
with mixed_fig:
    temperature_plot(potential, (x, -2, 2), (y, -2, 2), name="potential", label="x^2 + y^2 - 1", samples=40)
    contour_plot(potential, (x, -2, 2), (y, -2, 2), name="levels", label="levels", samples=40)
    domain_plot(-potential, (x, -2, 2), (y, -2, 2), name="inside", label="inside unit circle", samples=45)
    parametric_plot((cos(t), sin(t)), (t, 0, 2 * pi), name="circle", label="circle", samples=200)

## Shared and independent parameters

Superimposed plots can share some parameters while keeping others independent. Parameter identity is the SymPy symbol itself: layers that depend on the same symbol respond to the same slider, and layers that use a different symbol get their own control.

The next figure combines a heatmap, a contour layer, a signed domain, and a parametric boundary. The symbols `a` and `b` shape the field and contour together. The symbol `r` controls only the circular domain and boundary.

In [22]:
layered_fig = figure("plotting-guide-shared-independent-parameters")
layered_fig.show()

In [23]:
field = exp(-(x**2 + y**2) / 8) * cos(a * x + b * y)
disk = r**2 - x**2 - y**2

field_params = {
    a: {"value": 2.0, "min": 0.5, "max": 5.0, "step": 0.25},
    b: {"value": 1.0, "min": -3.0, "max": 3.0, "step": 0.25},
}
radius_params = {
    r: {"value": 1.25, "min": 0.5, "max": 2.0, "step": 0.05},
}
layered_fig.parameters({**field_params, **radius_params})

with layered_fig:
    temperature_plot(
        field,
        (x, -2.5, 2.5),
        (y, -2.5, 2.5),
        name="field",
        label="field value",
        samples=60,
        style={"colorscale": "Viridis", "opacity": 0.7},
    )
    contour_plot(
        field,
        (x, -2.5, 2.5),
        (y, -2.5, 2.5),
        name="field-levels",
        label="field levels",
        samples=60,
        style={"contour_color": "black", "contour_width": 1},
    )
    domain_plot(
        disk,
        (x, -2.5, 2.5),
        (y, -2.5, 2.5),
        name="disk",
        label="variable disk",
        samples=70,
        style={"domain": {"color": "white", "opacity": 0.35}, "boundary": {"color": "crimson", "width": 3}},
    )
    parametric_plot(
        (r * cos(t), r * sin(t)),
        (t, 0, 2 * pi),
        name="moving-boundary",
        label="radius boundary",
        samples=240,
        style={"color": "crimson", "width": 3, "dash": "dash"},
    )

## Styling

`style=` changes visual presentation without changing the mathematical expression. Line plots and parametric plots accept line-oriented keys such as `color`, `width`, `dash`, `opacity`, and `visible`.

Scalar-field plots accept field-oriented keys such as `colorscale`, `opacity`, and `showscale`. Domain plots use nested `domain` and `boundary` style sections.

In [24]:
style_fig = figure("plotting-guide-styling")
style_fig.show()

In [25]:
with style_fig:
    plot(sin(x), x, name="styled-sine", label="styled sine", style={"color": "crimson", "width": 3})
    plot(cos(x), x, name="styled-cosine", label="styled cosine", style={"color": "black", "dash": "dash"})

with style_fig:
    get_plot("styled-sine").style.opacity = 0.8

## Programmatic updates

Handles make interactive state available to notebook code. Assigning `fig.params` changes a figure parameter value or its slider metadata. Assigning `fig.view.range` changes the visible coordinate limits of the figure.

This section uses its own figure so the update effects stay local to the example.

In [26]:
programmatic_fig = figure("plotting-guide-programmatic-updates")
programmatic_fig.show()
programmatic_fig.parameters({a: {"value": 2.0, "min": 0.5, "max": 6.0, "step": 0.25}})

with programmatic_fig:
    plot(
        exp(-x**2 / 8) * cos(a * x),
        x,
        name="damped-wave",
        label="damped wave",
    )
    
programmatic_fig.params = {a: 4.0}
programmatic_fig.view.range = {"x": (-6, 6), "y": (-1.2, 1.2)}

## Views and programmatic recentering

A figure view is the coordinate window used by the plot area. It is not a second Plotly subplot or a separate panel; it is named view state that can be selected, resized, and reset from notebook code.

`fig.view.range` is the currently visible coordinate window. Assign it when a computation should pan or zoom the plot area. Use the dictionary shape `{"x": (xmin, xmax), "y": (ymin, ymax)}` when setting both axes, or assign `fig.view.x_range` and `fig.view.y_range` when changing one axis.

`fig.view.home_range` is the reset target. Plotly's home button and `fig.view.reset()` return to this rectangle. Use the same dictionary shape, or assign `fig.view.home_x_range` and `fig.view.home_y_range` for one axis at a time.

In [27]:
view_fig = figure("plotting-guide-views")
view_fig.show()

with view_fig:
    plot(sin(x) / (1 + x**2 / 16), (x, -12, 12), name="wave")

view_fig.view.range = {"x": (-8, 8), "y": (-0.5, 1.1)}
view_fig.view.home_range = {"x": (-8, 8), "y": (-0.5, 1.1)}

`fig.view("name")` returns a named `ViewHandle` for the same plot area. Named views are useful when a notebook switches between an overview and a close-up without rebuilding the plots. This section uses a fresh figure so the named-view state is independent of the previous recentering example.

In [28]:
named_view_fig = figure("plotting-guide-named-views")
named_view_fig.show()

with named_view_fig:
    plot(sin(x) / (1 + x**2 / 16), (x, -12, 12), name="wave")

overview = named_view_fig.view("overview")
closeup = named_view_fig.view("closeup")

overview.range = {"x": (-8, 8), "y": (-0.5, 1.1)}
closeup.range = {"x": (-2, 2), "y": (-0.1, 1.05)}

named_view_fig.view.current = "closeup"

## Display invalidation

A figure is durable, but each visible widget output is a display generation. By default, `fig.show()` creates a fresh live generation. The previous output remains visible in the notebook, but it is disconnected so its sliders, legend controls, Plotly pan and zoom events, and other frontend callbacks cannot keep changing the figure model.

This protects rerun-heavy notebooks: the newest displayed output is authoritative, while older outputs become snapshots of what was visible when they were retired. Use `fig.show(new=False)` only when you want to display the current live generation again instead of invalidating it and creating a new one.

In [29]:
invalidation_fig = figure("plotting-guide-invalidation")
invalidation_fig.show()
invalidation_fig.parameters({a: {"value": 1, "min": 0.5, "max": 3}})

with invalidation_fig:
    plot(sin(a * x), (x, -2 * pi, 2 * pi))

# A later display takes over as the live generation. The older visible output
# stays in the notebook, but it is no longer allowed to drive model updates.
invalidation_fig.show()

## Figure output area

A figure context routes ordinary notebook output into the figure's Output area. This is useful when a plot-producing cell also needs to leave a local note, a computed value, or a short trace of what it did.

The Output area starts collapsed. It expands when text is printed to it.

In [30]:
output_fig = figure("plotting-guide-output-area")
output_fig.show()

In [31]:
with output_fig:
    plot(sin(x), (x, -2 * pi, 2 * pi), name="sine", label="sin(x)")
    print("Sampled sin(x) over [-2*pi, 2*pi].")
    print("This text appears in the figure Output area, not as a separate cell output.")

When a cell prints more than the Output area's maximum height, the figure keeps a fixed-height output log and shows a scrollbar.

In [32]:
with output_fig:
    for n in range(1, 41):
        print(f"output line {n:02d}: scrollbar demonstration")

## Authored info cards

Figures can carry small live notes in the sidebar with `fig.info(...)`. Strings are Markdown. Symbols and expressions become live parameter-backed values, so changing a slider updates the note. Callables receive the figure and are useful for diagnostics that need current figure state.

Use info cards for figure-local context: current parameter choices, computed scalar summaries, warnings about interpretation, or a short note about what a comparison means. Status messages such as disconnected-generation notices remain separate from authored info.

In [33]:
information_fig = figure("plotting-guide-information-box")
information_fig.show()
information_fig.parameters({a: {"value": 1.0, "min": 0.25, "max": 4.0, "step": 0.25}})

with information_fig:
    plot(sin(a * x) / (1 + x**2), (x, -6, 6), name="message-demo")

information_fig.info(
    "For the current slider value, $a = ",
    a,
    "$ and the center value is ",
    lambda fig: fig.params[a]["value"],
    ".",
    name="summary",
    title="Live note",
)

## Custom layouts are advanced

Ordinary plotting work usually only needs `figure(...)`, `fig.show()`, figure contexts, named plots, and handles. Custom figure layouts arrange the plot widget, controls, and status area with an alternate Python-authored layout class; that is an advanced display topic rather than part of the core plotting workflow.

A dedicated layout tutorial is not available yet. For the current design reference, see [issue033: Design Python-authored plotting layout engine](../../../../docs/issues/issue033-design-python-authored-plotting-layout-engine.md).

## Recap

The plotting workflow starts with a figure, then uses a figure context for symbolic plotting commands. Plot names make later updates possible. Domains, parameters, styles, and handles refine what is sampled and how it appears.

`plot`, `list_plot`, `temperature_plot`, `contour_plot`, `domain_plot`, and `parametric_plot` cover the main symbolic plotting shapes. Numerically implemented functions enter the same workflow after being represented as symbolic `ImplementedFunction` objects.

## Related topics

- [figure](../library/figure.ipynb) for figure creation, display, routing, views, and layout hooks.
- [plot](../library/plot.ipynb) for scalar curves and plot handles.
- [list_plot](../library/list_plot.ipynb) for discrete point sets.
- [temperature_plot](../library/temperature_plot.ipynb) for heatmap scalar fields.
- [contour_plot](../library/contour_plot.ipynb) for level curves.
- [domain_plot](../library/domain_plot.ipynb) for Boolean and signed regions.
- [parametric_plot](../library/parametric_plot.ipynb) for coordinate-pair curves.
- [Num](../library/Num.ipynb) for the symbolic-to-numeric boundary.
- [Hamiltonian dynamics worksheet](../worksheets/hamiltonian_dynamics.ipynb) for plotting inside a longer numerical investigation.